In [1]:
import os
import wandb

print("WANDB_API_KEY exists:", os.getenv("WANDB_API_KEY") is not None)
print("WANDB_ENTITY:", os.getenv("WANDB_ENTITY"))
print("WANDB_PROJECT:", os.getenv("WANDB_PROJECT"))

ModuleNotFoundError: No module named 'wandb'

# Wandb smoketest

In [ ]:
run = wandb.init(
    entity=os.getenv("WANDB_ENTITY"),
    project=os.getenv("WANDB_PROJECT"),
    name="sanity-check"
)

wandb.log({"test_metric": 1})
run.finish()

In [ ]:
import torch
from src.baseline_forward import load_model, run_baseline
from src.rotation_utils import (
    random_orthogonal_matrix,
    hadamard_matrix,
    signed_hadamard_matrix,
    is_orthonormal,
    apply_rotation,
)

# Baseline

In [ ]:
model, tokenizer = load_model("gpt2")
results = run_baseline(model, tokenizer, text="Hello world")
baseline_logits = results["logits"]
activations = results["activations"]
print(baseline_logits.shape)
print(list(activations.keys()))

Activation statistics

In [ ]:
act = list(activations.values())[0]
print("shape:", act.shape)
print("mean:", act.mean().item())
print("std:", act.std().item())
print("max abs:", act.abs().max().item())

# Visualization

In [ ]:
import matplotlib.pyplot as plt
plt.hist(act.flatten().detach().cpu().numpy(), bins=100)
plt.title("Baseline activation distribution")
plt.show()

In [ ]:
act_dim = act.shape[-1]
R_rand_act = random_orthogonal_matrix(act_dim, device=act.device)

rot_act = apply_rotation(act, R_rand_act)

plt.hist(rot_act.flatten().detach().cpu().numpy(), bins=100)
plt.title("After Rotation")
plt.show()